# Experiment 7: Semantic Search + Extractive QA

In [49]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import torch

from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("Libraries imported successfully!")

Libraries imported successfully!


# Task 1: Semantic Search

## Aim
To find the most relevant document for a given query using true sentence embeddings and cosine similarity.

Five Data Science documents are used. Each document contains at least two paragraphs.

The `multi-qa-MiniLM-L6-cos-v1` model is used to generate sentence embeddings. Cosine similarity is then used to compare the query embedding with the document embeddings. The document with the highest similarity score is selected as the most relevant document.

In [50]:
documents = {}

for i in range(1, 6):
    filename = f"document{i}.txt"

    with open(filename, "r", encoding="utf-8") as file:
        documents[filename] = file.read()

print("Documents loaded successfully!")
print("Number of documents:", len(documents))

Documents loaded successfully!
Number of documents: 5


In [51]:
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

document_names = list(documents.keys())
document_texts = list(documents.values())
document_embeddings = embedding_model.encode(document_texts)

print("Document embeddings generated successfully!")
print("Number of documents:", len(document_embeddings))
print("Embedding size:", document_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Document embeddings generated successfully!
Number of documents: 5
Embedding size: (5, 384)


In [52]:
def semantic_search(question):
    
    question_embedding = embedding_model.encode([question])

    similarity_scores = cosine_similarity(
        question_embedding,
        document_embeddings
    )[0]

    best_index = similarity_scores.argmax()

    best_document_name = document_names[best_index]
    best_document = document_texts[best_index]
    best_score = similarity_scores[best_index]

    return best_document_name, best_document, best_score, similarity_scores

In [53]:
question = "What is data preprocessing?"

best_document_name, best_document, best_score, scores = semantic_search(question)

print("Question:", question)

print("\nSimilarity Scores:")
for name, score in zip(document_names, scores):
    print(f"{name}: {score:.4f}")

print("\nMost Relevant Document:")
print(best_document_name)

print("\nSimilarity Score:")
print(f"{best_score:.4f}")

Question: What is data preprocessing?

Similarity Scores:
document1.txt: 0.5396
document2.txt: 0.7768
document3.txt: 0.4699
document4.txt: 0.3896
document5.txt: 0.3729

Most Relevant Document:
document2.txt

Similarity Score:
0.7768


# Task 2: Extractive Question Answering

## Aim
To extract the answer from the most relevant document identified by semantic search.

The selected document from Task 1 is given as context to an extractive Question Answering model. The model extracts the answer directly from the document instead of generating a new answer.

The `distilbert-base-cased-distilled-squad` model is used for extractive Question Answering.

In [54]:
qa_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-cased-distilled-squad"
)

print("Extractive QA model loaded successfully!")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Extractive QA model loaded successfully!


In [55]:
def extract_answer(question, context):
    
    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop("offset_mapping")[0]
    sequence_ids = inputs.sequence_ids(0)

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_probs = torch.softmax(
        outputs.start_logits, dim=-1
    )[0]

    end_probs = torch.softmax(
        outputs.end_logits, dim=-1
    )[0]

    context_indices = [
        i for i, sid in enumerate(sequence_ids)
        if sid == 1
    ]

    best_score = 0
    best_start = context_indices[0]
    best_end = context_indices[0]

    for start in context_indices:
        for end in context_indices:
            
            if end < start:
                continue

            if end - start > 20:
                continue

            score = (
                start_probs[start].item()
                * end_probs[end].item()
            )

            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    start_char = offset_mapping[best_start][0].item()
    end_char = offset_mapping[best_end][1].item()

    answer = context[start_char:end_char]

    return answer, best_score

In [56]:
def answer_question(question):
    
    # Create embedding for the question
    question_embedding = embedding_model.encode([question])
    
    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        question_embedding,
        document_embeddings
    )[0]
    
    # Find the most relevant document
    best_index = similarity_scores.argmax()
    
    best_document_name = document_names[best_index]
    best_document = document_texts[best_index]
    best_similarity = similarity_scores[best_index]
    
    # Extract answer from the relevant document
    answer, qa_score = extract_answer(
        question,
        best_document
    )
    
    print("\n" + "=" * 60)
    print("QUESTION")
    print("=" * 60)
    print(question)
    
    print("\nMOST RELEVANT DOCUMENT")
    print("=" * 60)
    print(best_document_name)
    
    print("\nCOSINE SIMILARITY")
    print("=" * 60)
    print(f"{best_similarity:.4f}")
    
    print("\nEXTRACTED ANSWER")
    print("=" * 60)
    print(answer)
    
    print("\nQA SCORE")
    print("=" * 60)
    print(f"{qa_score:.4f}")

In [57]:
answer_question("What is data preprocessing?")


QUESTION
What is data preprocessing?

MOST RELEVANT DOCUMENT
document2.txt

COSINE SIMILARITY
0.7768

EXTRACTED ANSWER
an important step in Data Science

QA SCORE
0.5029
